# Reclamações 01 · Camada Silver — Referências

## 1. Objetivo e método

Este notebook produz as duas dimensões próprias do ranking de reclamações: `dim_tipologia` e `dim_tempo`. A terceira dimensão do modelo, `dim_distribuidora`, é conformada e já existe, criada em `base/04_silver_dimensoes`; ela é reaproveitada, não recriada.

### Por que a tipologia vem da norma, e não da base

A base de manifestações descreve a mesma tipologia com redações diferentes ao longo do tempo, e nenhuma tabela de apoio publicada pela ANEEL traz a hierarquia completa. A fonte escolhida é o Anexo I da Resolução Homologatória (REH) nº 2.992/2021, que homologa a tipologia de classificação das demandas: um código, uma descrição, uma posição na hierarquia. O anexo foi transcrito para `data/reference/reh_2992_tipologia.csv`, versionado no repositório, para que a dimensão seja reproduzível e auditável.

A partir de janeiro de 2024 o campo `CodTipoManifestacao` da base já usa o código da REH, o que dispensa qualquer tabela de-para. Por isso a série da Silver começa em 2024; 2022 e 2023 ficam na Bronze e são tratados apenas como achado de qualidade.

### Regras derivadas na dimensão

Todas as classificações usadas nos rankings nascem aqui, como atributos, para que a Silver e a Gold apenas filtrem:

| Atributo | Regra | Fonte |
|---|---|---|
| `ind_subtotal` | O código tem filhos na REH. Linhas de subtotal somariam em dobro com as detalhadas | Anexo I da REH 2.992 |
| `ind_fer_item285` | Família 102 (reclamação), sem subtotal, fora de interrupção (`1020901` a `1020903`), tensão (`1020904`) e ressarcimento de danos elétricos (grupo `10210`) | PRODIST Módulo 8, Seção 8.3, item 285 |
| `ind_comercial_estrito` | `ind_fer_item285`, sem Rede/Manutenção (grupo `10212`) e Outros de Qualidade (`1020999`) | Decisão do trabalho: esses dois grupos respondem por cerca de 74% das procedentes do item 285 e fariam o indicador medir causa técnica |
| `bloco` | Agrupamento usado na detecção de meses anômalos e na pergunta 5 | Combinação das regras acima |
| `grupo_ranking` | Recorte de cada ranking: Faturamento (`10204`), Pagamento/Inadimplência (`10206`), Geração Distribuída (`10213`) e Outras comerciais, dentro do comercial estrito, e Qualidade (`10209`), no técnico | Decisão do trabalho: grupos fixados pelo Pareto das procedentes; Faturamento e Pagamento afetam todas as classes de consumo. A Geração Distribuída ficou inicialmente em Outras, por atingir um grupo restrito de consumidores, e foi separada ao se tornar o 2º grupo do comercial estrito (17,8% das procedentes no LTM até jun/2026) e o principal motor das pioras em Outras. Conexão permanece em Outras pelo volume baixo por distribuidora |

O ranking que responde à pergunta 1 usa `ind_comercial_estrito`. O ranking pelo item 285 é apresentado ao lado, para tornar visível o efeito dos grupos técnicos.

### Ordem de execução

Este notebook roda depois de `base/04_silver_dimensoes` e antes de `complaints/02_silver_complaints`.

## 2. Configuração

In [ ]:
import os
import sys

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_SILVER

SILVER = f"{CATALOG}.{SCHEMA_SILVER}"

# Transcription of Annex I of REH 2.992/2021, versioned with the code
REH_CSV = os.path.join(REPO_ROOT, "data", "reference", "reh_2992_tipologia.csv")

# Complaint family and the exclusions of PRODIST Module 8, item 285
COD_FAMILIA_RECLAMACAO = "102"
COD_INTERRUPCAO = {"1020901", "1020902", "1020903"}
COD_TENSAO = {"1020904"}
GRUPO_DANOS_ELETRICOS = "10210"

# Level-2 groups that get their own ranking; the rest of the strict scope goes to "outras"
GRUPO_FATURAMENTO = "10204"
GRUPO_PAGAMENTO = "10206"
GRUPO_GERACAO_DISTRIBUIDA = "10213"
GRUPO_QUALIDADE = "10209"

# Groups removed from the item 285 scope to form the strict commercial scope
GRUPO_REDE_MANUTENCAO = "10212"
COD_OUTROS_QUALIDADE = {"1020999"}

# Silver time span: REH codes are used directly by the source from January 2024 on
ANO_MES_INICIO = 202401
ANO_MES_FIM = 202606

def para_registros(df, esquema):
    """Convert a pandas frame into plain Python records that match a Spark schema.

    Spark rejects numpy scalars in typed fields, and pandas carries empty strings and
    NaN where the dimension needs nulls, so every value is cast explicitly here.
    """
    tipos = {campo.name: campo.dataType for campo in esquema.fields}
    registros = []
    for linha in df.to_dict("records"):
        registro = {}
        for coluna, valor in linha.items():
            if valor is None or (isinstance(valor, str) and valor == "") or                     (not isinstance(valor, str) and pd.isna(valor)):
                registro[coluna] = None
            elif isinstance(tipos[coluna], T.BooleanType):
                registro[coluna] = bool(valor)
            elif isinstance(tipos[coluna], T.IntegerType):
                registro[coluna] = int(valor)
            else:
                registro[coluna] = valor
        registros.append(registro)
    return registros


spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_SILVER}")

print(f"Destino.........: {SILVER}")
print(f"Referencia REH..: {REH_CSV}")
print(f"Periodo.........: {ANO_MES_INICIO} a {ANO_MES_FIM}")

## 3. Referência da REH 2.992

#### Ler a transcrição do Anexo I:

O arquivo tem uma linha por código, com a descrição como publicada, o nível na hierarquia (1, 2 ou 3) e o código pai. O nível é dado pelo tamanho do código: três dígitos para o nível 1, cinco para o nível 2 e sete para o nível 3. Todos os campos são lidos como texto, para preservar os códigos como identificadores e não como números.

In [ ]:
reh = pd.read_csv(REH_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
reh["nivel"] = reh["nivel"].astype(int)

print(f"codigos na REH..: {len(reh)}")
print(reh.groupby("nivel").size().rename("codigos").to_string())

## 4. `dim_tipologia`

Grão: código de tipologia da REH. A dimensão cobre as 132 tipologias, inclusive as famílias que não são reclamação (informação, solicitação, denúncia, elogio, sugestão, cancelamento e encerramento). Assim, todo código da base encontra correspondência, e o que fica fora dos rankings sai por atributo, não por ausência na dimensão.

A derivação é feita em pandas, porque são 132 linhas e as regras dependem da hierarquia inteira. A tabela é gravada pelo Spark no Unity Catalog.

Valores de `bloco`:

| Valor | Conteúdo |
|---|---|
| `comercial_estrito` | Tipologias do recorte comercial estrito |
| `rede_e_qualidade_outros` | Rede/Manutenção (`10212`) e Outros de Qualidade (`1020999`): entram no item 285, mas não no recorte estrito |
| `tecnico` | Interrupção, tensão e ressarcimento de danos elétricos: excluídos pelo item 285 |
| `fora_da_familia_102` | Tipologias que não são reclamação |
| nulo | Linhas de subtotal, que não entram em nenhuma contagem |

Valores de `grupo_ranking`:

| Valor | Conteúdo |
|---|---|
| `faturamento` | Grupo `10204` Leitura/Faturamento/Fatura, no comercial estrito |
| `pagamento` | Grupo `10206` Pagamento/Inadimplência/Suspensão, no comercial estrito |
| `geracao_distribuida` | Grupo `10213` Geração Distribuída, no comercial estrito |
| `outras_comerciais` | Demais tipologias do comercial estrito, inclusive Conexão |
| `qualidade` | Grupo `10209` Qualidade: interrupção, tensão e outros de qualidade |
| nulo | Tipologias fora dos rankings e linhas de subtotal |

In [ ]:
codigos = set(reh["cod_tipologia"])
descricao = dict(zip(reh["cod_tipologia"], reh["descricao"]))

dim = reh.copy()
dim["cod_nivel_1"] = dim["cod_tipologia"].str[:3]
dim["cod_nivel_2"] = dim["cod_tipologia"].where(dim["nivel"] >= 2).str[:5]
dim["desc_nivel_1"] = dim["cod_nivel_1"].map(descricao)
dim["desc_nivel_2"] = dim["cod_nivel_2"].map(descricao)

# A code is a subtotal when some other code of the annex hangs below it
pais = set(reh["cod_pai"]) - {""}
dim["ind_subtotal"] = dim["cod_tipologia"].isin(pais)

folha_reclamacao = (dim["cod_nivel_1"] == COD_FAMILIA_RECLAMACAO) & ~dim["ind_subtotal"]
excluido_item285 = (dim["cod_tipologia"].isin(COD_INTERRUPCAO | COD_TENSAO)
                    | (dim["cod_nivel_2"] == GRUPO_DANOS_ELETRICOS))
rede_qualidade = ((dim["cod_nivel_2"] == GRUPO_REDE_MANUTENCAO)
                  | dim["cod_tipologia"].isin(COD_OUTROS_QUALIDADE))

dim["ind_fer_item285"] = folha_reclamacao & ~excluido_item285
dim["ind_comercial_estrito"] = dim["ind_fer_item285"] & ~rede_qualidade


def classificar_bloco(linha):
    # Subtotal rows never enter a count, so they carry no block
    if linha["ind_subtotal"]:
        return None
    if linha["cod_nivel_1"] != COD_FAMILIA_RECLAMACAO:
        return "fora_da_familia_102"
    if linha["ind_comercial_estrito"]:
        return "comercial_estrito"
    if linha["ind_fer_item285"]:
        return "rede_e_qualidade_outros"
    return "tecnico"


dim["bloco"] = dim.apply(classificar_bloco, axis=1)


def classificar_grupo_ranking(linha):
    # Subtotal rows never enter a count, so they belong to no ranking
    if linha["ind_subtotal"]:
        return None
    if linha["ind_comercial_estrito"]:
        if linha["cod_nivel_2"] == GRUPO_FATURAMENTO:
            return "faturamento"
        if linha["cod_nivel_2"] == GRUPO_PAGAMENTO:
            return "pagamento"
        if linha["cod_nivel_2"] == GRUPO_GERACAO_DISTRIBUIDA:
            return "geracao_distribuida"
        return "outras_comerciais"
    if linha["cod_nivel_2"] == GRUPO_QUALIDADE:
        return "qualidade"
    return None


dim["grupo_ranking"] = dim.apply(classificar_grupo_ranking, axis=1)

COLUNAS_DIM = ["cod_tipologia", "descricao", "nivel", "cod_pai",
               "cod_nivel_1", "desc_nivel_1", "cod_nivel_2", "desc_nivel_2",
               "ind_subtotal", "ind_fer_item285", "ind_comercial_estrito", "bloco",
               "grupo_ranking"]

esquema_dim = T.StructType([
    T.StructField("cod_tipologia", T.StringType(), False),
    T.StructField("descricao", T.StringType(), False),
    T.StructField("nivel", T.IntegerType(), False),
    T.StructField("cod_pai", T.StringType(), True),
    T.StructField("cod_nivel_1", T.StringType(), False),
    T.StructField("desc_nivel_1", T.StringType(), False),
    T.StructField("cod_nivel_2", T.StringType(), True),
    T.StructField("desc_nivel_2", T.StringType(), True),
    T.StructField("ind_subtotal", T.BooleanType(), False),
    T.StructField("ind_fer_item285", T.BooleanType(), False),
    T.StructField("ind_comercial_estrito", T.BooleanType(), False),
    T.StructField("bloco", T.StringType(), True),
    T.StructField("grupo_ranking", T.StringType(), True),
])

dim_tipologia = spark.createDataFrame(para_registros(dim[COLUNAS_DIM], esquema_dim),
                                      schema=esquema_dim)

(dim_tipologia.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_tipologia"))

print(f"tipologias..................: {dim_tipologia.count()}")
print(f"subtotais...................: {dim_tipologia.filter('ind_subtotal').count()}")
print(f"recorte item 285............: {dim_tipologia.filter('ind_fer_item285').count()}")
print(f"recorte comercial estrito...: {dim_tipologia.filter('ind_comercial_estrito').count()}")

display(dim_tipologia
        .groupBy("bloco", "cod_nivel_2", "desc_nivel_2")
        .agg(F.count("*").alias("tipologias"))
        .orderBy("bloco", "cod_nivel_2"))

In [ ]:
COMENTARIOS_TIPOLOGIA = {
    "cod_tipologia": "Codigo da tipologia conforme Anexo I da REH 2.992/2021, texto",
    "descricao": "Descricao da tipologia conforme publicada no Anexo I da REH 2.992/2021",
    "nivel": "Nivel na hierarquia da REH: 1 (familia), 2 (grupo) ou 3 (tipologia)",
    "cod_pai": "Codigo imediatamente superior na hierarquia; nulo no nivel 1",
    "cod_nivel_1": "Codigo da familia (nivel 1) a que a tipologia pertence",
    "desc_nivel_1": "Descricao da familia (nivel 1)",
    "cod_nivel_2": "Codigo do grupo (nivel 2) a que a tipologia pertence; nulo no nivel 1",
    "desc_nivel_2": "Descricao do grupo (nivel 2); nulo no nivel 1",
    "ind_subtotal": "Verdadeiro quando o codigo tem filhos na REH; linha de subtotal, fora de toda contagem",
    "ind_fer_item285": "Verdadeiro para reclamacao detalhada que entra no FER pela regra do PRODIST Modulo 8, item 285",
    "ind_comercial_estrito": "Recorte do item 285 sem Rede e Manutencao (10212) e Outros de Qualidade (1020999)",
    "bloco": "Agrupamento para deteccao de meses anomalos e analise: comercial_estrito, rede_e_qualidade_outros, tecnico ou fora_da_familia_102",
    "grupo_ranking": "Recorte do ranking: faturamento (10204), pagamento (10206), geracao_distribuida (10213) ou outras_comerciais no comercial estrito, e qualidade (10209); nulo fora dos rankings",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.dim_tipologia IS
    'Dimensao de tipologias de manifestacao conforme Anexo I da REH 2.992/2021. Carrega a
     hierarquia de tres niveis e os recortes usados nos rankings como atributos; a Silver
     e a Gold apenas filtram por eles.'""")

for coluna, texto in COMENTARIOS_TIPOLOGIA.items():
    spark.sql(f"ALTER TABLE {SILVER}.dim_tipologia "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_TIPOLOGIA)} colunas")

## 5. `dim_tempo`

Grão: ano-mês. A série vai de janeiro de 2024 a junho de 2026, último mês consolidado nas fontes. O atributo `ind_fim_janela` marca junho e dezembro, os meses em que termina cada janela móvel de 12 meses usada no acompanhamento semestral; com a série iniciando em janeiro de 2024, a primeira janela completa termina em dezembro de 2024.

In [ ]:
meses = pd.period_range(start=pd.Period(str(ANO_MES_INICIO), freq="M"),
                        end=pd.Period(str(ANO_MES_FIM), freq="M"), freq="M")

tempo = pd.DataFrame({
    "ano_mes": [p.year * 100 + p.month for p in meses],
    "ano": [p.year for p in meses],
    "mes": [p.month for p in meses],
    "data_inicio_mes": [p.start_time.date() for p in meses],
})
tempo["semestre"] = (tempo["mes"] > 6).astype(int) + 1
tempo["ind_fim_janela"] = tempo["mes"].isin([6, 12])

esquema_tempo = T.StructType([
    T.StructField("ano_mes", T.IntegerType(), False),
    T.StructField("ano", T.IntegerType(), False),
    T.StructField("mes", T.IntegerType(), False),
    T.StructField("data_inicio_mes", T.DateType(), False),
    T.StructField("semestre", T.IntegerType(), False),
    T.StructField("ind_fim_janela", T.BooleanType(), False),
])

dim_tempo = spark.createDataFrame(
    para_registros(tempo[[f.name for f in esquema_tempo.fields]], esquema_tempo),
    schema=esquema_tempo)

(dim_tempo.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_tempo"))

print(f"meses...........: {dim_tempo.count()}")
print(f"fins de janela..: {[r['ano_mes'] for r in dim_tempo.filter('ind_fim_janela').orderBy('ano_mes').collect()]}")

In [ ]:
COMENTARIOS_TEMPO = {
    "ano_mes": "Ano e mes de competencia no formato AAAAMM, chave da dimensao",
    "ano": "Ano de competencia",
    "mes": "Mes de competencia, de 1 a 12",
    "data_inicio_mes": "Primeiro dia do mes de competencia",
    "semestre": "Semestre do ano: 1 (janeiro a junho) ou 2 (julho a dezembro)",
    "ind_fim_janela": "Verdadeiro em junho e dezembro, meses em que termina uma janela movel de 12 meses do acompanhamento semestral",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.dim_tempo IS
    'Dimensao de tempo mensal da Silver de reclamacoes, de janeiro de 2024 a junho de 2026.
     Marca os fins de janela do acompanhamento semestral.'""")

for coluna, texto in COMENTARIOS_TEMPO.items():
    spark.sql(f"ALTER TABLE {SILVER}.dim_tempo "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_TEMPO)} colunas")

## 6. Validação

As contagens esperadas vêm da própria norma: o Anexo I tem 132 códigos; a família 102 tem 83 tipologias detalhadas, das quais o item 285 exclui 10 (três de interrupção, uma de tensão e seis de danos elétricos), restando 73; o recorte estrito retira mais 8 (sete de Rede/Manutenção e Outros de Qualidade), restando 65. Se a transcrição ou a regra estiverem erradas, algum desses números não fecha.

In [ ]:
tip = spark.table(f"{SILVER}.dim_tipologia")
tmp = spark.table(f"{SILVER}.dim_tempo")

testes = []

linhas = tip.count()
unicas = tip.select("cod_tipologia").distinct().count()
testes.append(("chave unica por tipologia",
               linhas == unicas,
               f"{linhas} linhas para {unicas} codigos distintos"))

testes.append(("transcricao completa do Anexo I",
               linhas == 132,
               f"{linhas} codigos, esperados 132"))

orfaos = (tip.filter(F.col("cod_pai").isNotNull()).alias("f")
          .join(tip.alias("p"), F.col("f.cod_pai") == F.col("p.cod_tipologia"), "left_anti")
          .count())
testes.append(("todo codigo tem pai na REH",
               orfaos == 0,
               f"{orfaos} codigos com pai inexistente"))

folhas_102 = tip.filter((F.col("cod_nivel_1") == "102") & ~F.col("ind_subtotal")).count()
testes.append(("tipologias detalhadas da familia 102",
               folhas_102 == 83,
               f"{folhas_102}, esperadas 83"))

item285 = tip.filter("ind_fer_item285").count()
testes.append(("recorte do item 285",
               item285 == 73,
               f"{item285}, esperadas 73"))

estrito = tip.filter("ind_comercial_estrito").count()
testes.append(("recorte comercial estrito",
               estrito == 65,
               f"{estrito}, esperadas 65"))

fora = tip.filter(F.col("ind_comercial_estrito") & ~F.col("ind_fer_item285")).count()
testes.append(("estrito contido no item 285",
               fora == 0,
               f"{fora} tipologias no estrito fora do item 285"))

subtotal_marcado = tip.filter(F.col("ind_subtotal") &
                              (F.col("ind_fer_item285") | F.col("bloco").isNotNull())).count()
testes.append(("subtotal fora de todo recorte",
               subtotal_marcado == 0,
               f"{subtotal_marcado} subtotais com recorte ou bloco"))

sem_bloco = tip.filter(~F.col("ind_subtotal") & F.col("bloco").isNull()).count()
testes.append(("toda tipologia detalhada tem bloco",
               sem_bloco == 0,
               f"{sem_bloco} tipologias detalhadas sem bloco"))

grupos = {r["grupo_ranking"]: r["count"] for r in
          tip.filter(F.col("grupo_ranking").isNotNull()).groupBy("grupo_ranking").count().collect()}
esperado_grupos = {"faturamento": 15, "pagamento": 7, "geracao_distribuida": 5, "outras_comerciais": 38, "qualidade": 5}
testes.append(("tipologias por grupo de ranking",
               grupos == esperado_grupos,
               f"{dict(sorted(grupos.items()))}"))

grupos_comerciais = tip.filter(F.col("grupo_ranking").isin("faturamento", "pagamento", "geracao_distribuida",
                                                            "outras_comerciais"))
fora_estrito = grupos_comerciais.filter(~F.col("ind_comercial_estrito")).count()
testes.append(("grupos comerciais cobrem o estrito",
               fora_estrito == 0 and grupos_comerciais.count() == estrito,
               f"{grupos_comerciais.count()} tipologias nos grupos, {estrito} no estrito"))

meses = tmp.count()
testes.append(("meses de jan/2024 a jun/2026",
               meses == 30,
               f"{meses} meses, esperados 30"))

fins = [r["ano_mes"] for r in tmp.filter("ind_fim_janela").orderBy("ano_mes").collect()]
testes.append(("fins de janela semestrais",
               fins == [202406, 202412, 202506, 202512, 202606],
               f"{fins}"))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<45} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nReferencias validadas.")
else:
    print("\nHa teste sem passar; corrigir antes de seguir para a Silver de reclamacoes.")

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Códigos da base fora da REH | Esperados em 2025 (`1021307`, `1020910`, `1021101`, `1021102`) | Contados e excluídos em `complaints/02_silver_complaints` |
| Recorte comercial estrito | Decisão do trabalho, não da norma | Declarado nos disclaimers do README e da autoavaliação |

## Autoavaliação desta etapa

A preencher após a execução.